Exploratory Data Analysis: Western and Asian Folktales

This notebook explores the ATU-labeled Western folktales and collects Asian/Southeast Asian folktales from Project Gutenberg.

Goals:
1. Load and inspect Western folktales with ATU labels
2. Collect Asian/SE Asian folktales
3. Compare text statistics between datasets
4. Visualize ATU category distribution
5. Check data quality and identify preprocessing needs

In [ ]:
# Import libraries
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import yaml
import logging
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import FolktaleDataLoader

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Load configuration
with open('../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print('Configuration loaded.')

Load Western Folktales with ATU Labels

In [ ]:
# Initialize data loader
loader = FolktaleDataLoader('../config/config.yaml')

# Try to load Western tales (you'll need to download aft.csv from the GitHub link)
try:
    western_df = loader.load_western_tales()
    print(f"Loaded {len(western_df)} Western folktales")
    print(f"\nColumns: {western_df.columns.tolist()}")
    print(f"\nFirst row:")
    print(western_df.iloc[0])
except FileNotFoundError as e:
    print(f"Note: {e}")
    print("\nPlease download the AFT dataset from:")
    print("https://github.com/j-hagedorn/trilogy/blob/master/data/aft.csv")
    print("\nAnd place it at: data/western/aft.csv")
    western_df = None

Text Statistics for Western Folktales

In [ ]:
if western_df is not None and 'text' in western_df.columns:
    # Compute text statistics
    western_df['text_length'] = western_df['text'].str.len()
    western_df['word_count'] = western_df['text'].str.split().str.len()
    western_df['sentence_count'] = western_df['text'].str.count(r'\.')
    
    # Display statistics
    print("Western Folktales Text Statistics:")
    print(western_df[['text_length', 'word_count', 'sentence_count']].describe())
    
    # Plot distributions
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    axes[0].hist(western_df['text_length'], bins=50, color='blue', alpha=0.7, edgecolor='black')
    axes[0].set_xlabel('Text Length (characters)')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Western Tales: Text Length Distribution')
    
    axes[1].hist(western_df['word_count'], bins=50, color='green', alpha=0.7, edgecolor='black')
    axes[1].set_xlabel('Word Count')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Western Tales: Word Count Distribution')
    
    axes[2].hist(western_df['sentence_count'], bins=50, color='red', alpha=0.7, edgecolor='black')
    axes[2].set_xlabel('Sentence Count')
    axes[2].set_ylabel('Frequency')
    axes[2].set_title('Western Tales: Sentence Count Distribution')
    
    plt.tight_layout()
    plt.savefig('../results/plots/01_western_text_stats.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Plot saved.")

ATU Category Distribution

In [ ]:
if western_df is not None and 'atu_category' in western_df.columns:
    # Get unique ATU categories
    atu_counts = western_df['atu_category'].value_counts()
    
    print(f"Number of unique ATU categories: {len(atu_counts)}")
    print(f"\nTop 20 ATU categories:")
    print(atu_counts.head(20))
    
    # Plot top categories
    fig, ax = plt.subplots(figsize=(14, 6))
    atu_counts.head(20).plot(kind='barh', ax=ax, color='steelblue', edgecolor='black')
    ax.set_xlabel('Number of Tales')
    ax.set_title('Top 20 ATU Categories by Frequency')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig('../results/plots/01_atu_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("ATU distribution plot saved.")

Load Asian/Southeast Asian Folktales

The Asian folktales are stored as JSON files in the parent directory (Downloads folder):
- `china_china_fables_dataset (1).json`
- `korea_korea_fables_dataset.json` 
- `japan_japan_fables_dataset.json`

In [ ]:
# Try to load Asian folktales
try:
    asian_df = loader.load_asian_tales()
    if len(asian_df) > 0:
        print(f"Loaded {len(asian_df)} Asian/SE Asian folktales")
        print(f"\nRegions: {asian_df['region'].value_counts()}")
        print(f"\nFirst tale preview:")
        print(asian_df.iloc[0]['text'][:500])
    else:
        print("No Asian folktales found.")
        print("\nExpected JSON files in parent directory (Downloads folder):")
        print("- china_china_fables_dataset (1).json")
        print("- korea_korea_fables_dataset.json")
        print("- japan_japan_fables_dataset.json")
        print("\nEach JSON should contain an array of tale objects with 'text' field")
        asian_df = None
except Exception as e:
    print(f"Error loading Asian folktales: {e}")
    asian_df = None

Asian Folktales Statistics

In [ ]:
if asian_df is not None and len(asian_df) > 0:
    # Compute text statistics
    asian_df['text_length'] = asian_df['text'].str.len()
    asian_df['word_count'] = asian_df['text'].str.split().str.len()
    asian_df['sentence_count'] = asian_df['text'].str.count(r'\.')
    
    # Display statistics
    print("Asian Folktales Text Statistics:")
    print(asian_df[['text_length', 'word_count', 'sentence_count']].describe())
    
    # Plot distributions
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    axes[0].hist(asian_df['text_length'], bins=50, color='orange', alpha=0.7, edgecolor='black')
    axes[0].set_xlabel('Text Length (characters)')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Asian Tales: Text Length Distribution')
    
    axes[1].hist(asian_df['word_count'], bins=50, color='red', alpha=0.7, edgecolor='black')
    axes[1].set_xlabel('Word Count')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Asian Tales: Word Count Distribution')
    
    axes[2].hist(asian_df['sentence_count'], bins=50, color='purple', alpha=0.7, edgecolor='black')
    axes[2].set_xlabel('Sentence Count')
    axes[2].set_ylabel('Frequency')
    axes[2].set_title('Asian Tales: Sentence Count Distribution')
    
    plt.tight_layout()
    plt.savefig('../results/plots/01_asian_text_stats.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Asian statistics plot saved.")

Summary and Next Steps

In [ ]:
print(f"Western folktales: {len(western_df) if western_df is not None else 'Not loaded'}")
print(f"Asian/SE Asian folktales: {len(asian_df) if asian_df is not None and len(asian_df) > 0 else 'Not loaded'}")
print("Next: Run notebook 02_classifier.ipynb to train the logistic regression classifier")
print("  on Western folktales and test confidence on Asian folktales.")